# 패스트볼 K-Means 클러스터링 — 결론 정리 (step6)

**분석 단위**: 투수-시즌(투수-시즌별 주 패스트볼 프로필, `fastball_profile_table.pkl`)

**목표**: 투수-시즌별 주 패스트볼 특성(구속, 무브먼트, 릴리스 정보)을 바탕으로 K-Means 클러스터링을 수행하고,
결과가 실제 구종 분류(FF/SI/FC)와 얼마나 일치하는지 검증해서 클러스터의 의미를 확정한다.

**최종 결론 미리보기**: elbow/silhouette 지표는 k=2가 가장 높았지만, k=3의 클러스터별 평균 프로필을 보면
팔각도 ↔ 무브먼트 방향(상하 vs 좌우)의 물리적 관계가 뚜렷하게 드러나 **k=3이 해석 가능한 구조**로 판단됨.
아래에서 실제 구종(primary_fastball_type)과 교차표로 이를 검증한다.

## 0. 환경 설정 및 데이터 로드

In [13]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

# 한글 폰트 설정 (제목/라벨 깨짐 방지)
for _font_name in ("AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
    if _font_name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _font_name
        break
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.precision", 2)

In [14]:
def _find_repo_root() -> Path:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "data").is_dir():
            return candidate
    return start


ROOT = _find_repo_root()
INTERIM_DIR = ROOT / "data" / "processed" / "interim"
OUTPUT_DIR = ROOT / "data" / "processed" / "output"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

FEATURE_COLS = [
    "release_speed",
    "ivb_ft",
    "hb_ft",
    "arm_angle",
    "release_pos_x",
    "release_pos_z",
    "release_extension",
]

RANDOM_STATE = 42

profile = pd.read_pickle(INTERIM_DIR / "fastball_profile_table.pkl")
print(f"투수-시즌 행 수: {len(profile):,}")
profile.head()

투수-시즌 행 수: 1,860


,pitcher,player_name,game_year,primary_fastball_type,release_speed,ivb_ft,hb_ft,arm_angle,release_pos_x,release_pos_z,release_extension
0,425794,"Wainwright, Adam",2021,SI,89.06,11.30,11.72,44.74,-1.21,6.21,6.56
1,425794,"Wainwright, Adam",2022,SI,88.57,11.01,11.81,44.24,-1.12,6.23,6.51
2,425794,"Wainwright, Adam",2023,SI,86.87,9.81,11.55,43.35,-1.26,6.13,6.55
3,425844,"Greinke, Zack",2021,FF,88.94,14.61,1.15,48.84,-1.22,6.41,5.99
4,425844,"Greinke, Zack",2022,FF,89.13,14.12,1.81,44.01,-1.69,6.25,5.93


## 1. k 후보 비교 (elbow / silhouette + PCA 시각화)

k=2~10 범위에서 inertia/silhouette를 계산하고, k=2/3/4를 PCA 2D로 나란히 비교한다.
- silhouette만 보면 k=2가 가장 높음(0.249)
- 하지만 k=4에서 왼쪽 끝에 떨어져 있던 소수 투수-시즌이 별도 클러스터로 분리되는 걸 확인 — 이상치 그룹 존재 가능성
- k=3은 큰 덩어리를 둘로 나누면서도 해석 가능한 팔각도/무브먼트 관계를 보여줌 (2절에서 검증)

In [15]:
X = profile[FEATURE_COLS].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_candidates = range(2, 11)
rows = []
for k in k_candidates:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": sil})

k_table = pd.DataFrame(rows)
k_table.to_csv(OUTPUT_DIR / "kmeans_k_selection.csv", index=False)
k_table

,k,inertia,silhouette
0,2,10210.30,0.25
1,3,8983.57,0.17
2,4,8004.27,0.19
3,5,7166.91,0.19
4,6,6554.94,0.19
5,7,6076.90,0.17
6,8,5678.48,0.18
7,9,5388.82,0.18
8,10,5122.32,0.17


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(k_table["k"], k_table["inertia"], marker="o")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inertia")
axes[0].set_title("Elbow")

axes[1].plot(k_table["k"], k_table["silhouette"], marker="o", color="darkorange")
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette score")
axes[1].set_title("Silhouette")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_k_selection.png", dpi=150)
plt.show()

/var/folders/63/blb90k956gg_3ngmcpxm_0_c0000gn/T/ipykernel_18160/1106733129.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
compare_ks = [2, 3, 4]
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.sum()

fig, axes = plt.subplots(1, len(compare_ks), figsize=(6 * len(compare_ks), 5.5))
for ax, k in zip(axes, compare_ks):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", s=20, alpha=0.8)
    ax.set_title(f"k={k} (silhouette={sil:.3f})")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

fig.suptitle(f"k 비교 (PCA 2D, 설명된 분산 {explained:.1%})")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_k_comparison.png", dpi=150)
plt.show()

/var/folders/63/blb90k956gg_3ngmcpxm_0_c0000gn/T/ipykernel_18160/373079387.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. 최종 k=3 모델 학습 및 클러스터 프로파일링

k=3으로 최종 학습하고, 각 클러스터의 평균 특성을 확인한다.

In [18]:
FINAL_K = 3

final_km = KMeans(n_clusters=FINAL_K, n_init=10, random_state=RANDOM_STATE)
labels = final_km.fit_predict(X_scaled)
sil_final = silhouette_score(X_scaled, labels)
sample_sil = silhouette_samples(X_scaled, labels)

profile["cluster"] = labels
profile["silhouette_sample"] = sample_sil

print(f"최종 k={FINAL_K}, silhouette={sil_final:.4f}")
profile["cluster"].value_counts().sort_index()

최종 k=3, silhouette=0.1728


cluster
0    558
1    485
2    817
Name: count, dtype: int64

In [19]:
cluster_profile = profile.groupby("cluster")[FEATURE_COLS].agg(["mean", "std"]).round(2)
cluster_profile.to_csv(OUTPUT_DIR / "kmeans_cluster_profile.csv")
cluster_profile

release_speed       ivb_ft        hb_ft       arm_angle         \
                 mean   std   mean   std   mean   std      mean    std   
cluster                                                                  
0               92.09  2.38  14.08  3.85   5.56  5.38     45.91   8.56   
1               93.52  2.69   7.89  4.89  14.54  3.33     25.49  14.81   
2               95.32  1.92  16.87  2.24   7.76  3.61     42.73   8.76   

        release_pos_x       release_pos_z       release_extension        
                 mean   std          mean   std              mean   std  
cluster                                                                  
0                0.24  1.75          6.08  0.36              6.18  0.36  
1               -0.67  2.31          5.35  0.71              6.38  0.44  
2               -1.43  1.22          5.89  0.36              6.63  0.35

**해석**

| 클러스터 | 구속 | IVB(상하) | HB(좌우) | 팔각도 | 특징 |
|---|---|---|---|---|---|
| 0 | 92.1 | 14.1 | 5.6 | 45.9 | 일반 라이징 포심 |
| 1 | 93.5 | 7.9 (낮음) | 14.5 (높음) | 25.5 (낮음) | 싱커/컷터 계열 — 팔각도 낮아 좌우 무브먼트 큼 |
| 2 | 95.3 (빠름) | 16.9 (높음) | 7.8 | 42.7 | 엘리트급 라이징 포심 |

팔각도가 낮을수록 좌우 움직임(HB)이 커지고 상하 움직임(IVB)이 작아지는 물리적 관계가 뚜렷하게 나타남
→ 단순히 분포를 억지로 자른 것이 아니라 의미 있는 구조로 판단됨.

In [20]:
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", s=25, alpha=0.8)
ax.set_title(f"K-Means Clusters (k={FINAL_K}) — PCA 2D\n설명된 분산: {explained:.1%}")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.colorbar(scatter, label="cluster")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_pca_clusters.png", dpi=150)
plt.show()

/var/folders/63/blb90k956gg_3ngmcpxm_0_c0000gn/T/ipykernel_18160/1294093845.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 검증: 클러스터 vs 실제 구종(FF/SI/FC) 교차표

클러스터 1이 실제로 싱커(SI)/컷터(FC) 투수들과 겹치는지 확인한다.
일치도가 높으면 위 해석(팔각도 ↔ 무브먼트 방향)이 데이터로 뒷받침되는 것.

In [21]:
crosstab = pd.crosstab(
    profile["cluster"], profile["primary_fastball_type"],
    margins=True, margins_name="합계",
)
crosstab

primary_fastball_type,FC,FF,SI,합계
cluster,,,,
0,88,403,67,558
1,1,146,338,485
2,17,743,57,817
합계,106,1292,462,1860


In [22]:
crosstab_pct = pd.crosstab(
    profile["cluster"], profile["primary_fastball_type"],
    normalize="index",
).round(3) * 100
crosstab_pct.columns = [f"{c} (%)" for c in crosstab_pct.columns]
crosstab_pct

,FC (%),FF (%),SI (%)
cluster,,,
0,15.8,72.2,12.0
1,0.2,30.1,69.7
2,2.1,90.9,7.0


**교차표 해석 가이드**
- 클러스터 1의 행에서 SI/FC 비율이 다른 클러스터보다 뚜렷이 높다면 → "팔각도 낮음 = 싱커/컷터 계열" 해석이 검증됨
- 만약 세 클러스터 모두 FF 비율이 압도적으로 높고 구종 구분과 무관하다면 → 클러스터가 구종이 아니라 같은 포심 내에서의
  "스타일 차이"(라이징 vs 일반)를 포착한 것으로 재해석 필요

## 4. 다른 팀원 결과(계층적 / DBSCAN / GNN)와 비교할 지표 저장

동일한 기준(실루엣 점수, 클러스터 수, 클러스터별 크기, 노이즈 처리 여부)으로 정리해서
발표 시 다른 클러스터링 기법과 나란히 비교할 수 있게 한다.

In [23]:
import json

sizes = profile["cluster"].value_counts().sort_index()
metrics = {
    "method": "KMeans",
    "unit": "pitcher-season (primary fastball)",
    "n_clusters": int(FINAL_K),
    "silhouette_score": round(float(sil_final), 4),
    "cluster_sizes": {int(c): int(n) for c, n in sizes.items()},
    "noise_points": 0,
    "notes": (
        "K-Means는 구형(spherical) 클러스터를 가정하고 k를 사전에 지정해야 함. "
        "DBSCAN과 달리 이상치를 별도 노이즈로 분리하지 않고 모든 투수-시즌을 "
        "클러스터에 할당함. 계층적/GNN 결과와 클러스터 수·크기 비교 시 참고."
    ),
}

with open(OUTPUT_DIR / "kmeans_comparison_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

metrics

{'method': 'KMeans',
 'unit': 'pitcher-season (primary fastball)',
 'n_clusters': 3,
 'silhouette_score': 0.1728,
 'cluster_sizes': {0: 558, 1: 485, 2: 817},
 'noise_points': 0,
 'notes': 'K-Means는 구형(spherical) 클러스터를 가정하고 k를 사전에 지정해야 함. DBSCAN과 달리 이상치를 별도 노이즈로 분리하지 않고 모든 투수-시즌을 클러스터에 할당함. 계층적/GNN 결과와 클러스터 수·크기 비교 시 참고.'}

## 5. 최종 결과 저장

In [24]:
out_path = OUTPUT_DIR / "fastball_clusters_final.csv"
profile.to_csv(out_path, index=False)
print(f"저장 완료: {out_path} ({len(profile):,}행)")

저장 완료: /Users/yoon-uijin/Documents/GitHub/9th-first-project-baseball-1/data/processed/output/fastball_clusters_final.csv (1,860행)


## 결론 요약

1. **k 결정**: silhouette 지표만으로는 k=2가 최적이지만, k=3에서 클러스터별 프로필이 팔각도-무브먼트 방향의
   물리적 관계를 뚜렷하게 반영하여 **k=3을 최종 채택**함.
2. **클러스터 해석**:
   - 클러스터 0: 일반 라이징 포심 (팔각도 높음, IVB 중간)
   - 클러스터 1: 싱커/컷터 계열 (팔각도 낮음, HB 큼, IVB 작음)
   - 클러스터 2: 엘리트급 라이징 포심 (구속·IVB 모두 최고)
3. **검증**: `primary_fastball_type`과의 교차표로 위 해석이 실제 구종 분류와 얼마나 일치하는지 확인함 (3절 참고).
4. **비교 지표**: 계층적/DBSCAN/GNN 결과와 비교할 수 있도록 실루엣 점수, 클러스터 수/크기, 노이즈 처리 여부를
   `kmeans_comparison_metrics.json`으로 정리함.
5. **산출물**:
   - `fastball_clusters_final.csv` — 투수-시즌별 클러스터 라벨
   - `kmeans_cluster_profile.csv` — 클러스터별 평균/표준편차
   - `kmeans_pca_clusters.png`, `kmeans_k_comparison.png` — 시각화
   - `kmeans_comparison_metrics.json` — 팀 비교용 지표